# 06: Downstream Sentiment Probe (Twi Only)

**Objective**: Perform a lightweight, honest downstream check to investigate whether lower token fertility from GhanaTok translates into measurable differences on a downstream task (sentiment classification on the public AfriSenti Twi dataset).

> **Scope & Framing Note**: This is a lightweight exploratory probe using a bag-of-tokens Logistic Regression classifier across all tokenizers to ensure the comparison is fair and strictly about tokenization characteristics. It is not a full deep learning downstream benchmark and should be interpreted as a directional signal.

## 1. Load AfriSenti Twi Dataset

In [1]:
from datasets import load_dataset
import pandas as pd

# Load AfriSenti Twi dataset
afrisenti = load_dataset("masakhane/afrisenti", "twi")
print(afrisenti)
print("Sample train example:", afrisenti["train"][0])

DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 3481
    })
    validation: Dataset({
        features: ['tweet', 'label'],
        num_rows: 388
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 949
    })
})
Sample train example: {'tweet': 'kako be shark but wo ti ewu', 'label': 'negative'}


## 2. Load All Tokenizers
We load baseline tokenizers (mBERT, XLM-R, AfroXLMR), the regional baseline (ABENA), and both GhanaTok variants (BPE & SentencePiece Unigram).

In [2]:
import sys
from pathlib import Path
from transformers import PreTrainedTokenizerFast

sys.path.append("..")
from src.tokenizer_benchmark import load_tokenizers, load_twi_only_tokenizers
from src.train_ghanatok import SentencePieceWrapper

baseline_tokenizers = load_tokenizers()
twi_only_tokenizers = load_twi_only_tokenizers()

ghanatok = PreTrainedTokenizerFast(
    tokenizer_file="../models/ghanatok/ghanatok_tokenizer.json",
    unk_token="<unk>", pad_token="<pad>", bos_token="<s>", eos_token="</s>",
)
ghanatok_unigram = SentencePieceWrapper("../models/ghanatok_unigram/ghanatok_unigram.model")

all_probe_tokenizers = {
    **baseline_tokenizers,
    "ABENA": twi_only_tokenizers["ABENA"],
    "GhanaTok": ghanatok,
    "GhanaTok-Unigram": ghanatok_unigram,
}

Loading mBERT (bert-base-multilingual-cased) ...
Loading XLM-R (xlm-roberta-base) ...
Loading AfroXLMR (Davlan/afro-xlmr-large-29L) ...
Loading ABENA (Ghana-NLP/abena-base-akuapem-twi-cased) ...


## 3. Train Bag-of-Tokens Logistic Regression Probe
Using `CountVectorizer` wrapped with each tokenizer's `.tokenize()` analyzer.

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

def tokenizer_to_analyzer(tokenizer):
    """Wraps any tokenizer so CountVectorizer can use it as a text splitter."""
    return lambda text: tokenizer.tokenize(text)

train_data = afrisenti["train"]
text_col = "tweet" if "tweet" in train_data.column_names else "text"
texts = train_data[text_col]
labels = train_data["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

probe_results = []
for name, tok in all_probe_tokenizers.items():
    vectorizer = CountVectorizer(analyzer=tokenizer_to_analyzer(tok))
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)
    
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    
    probe_results.append({
        "tokenizer": name,
        "vocab_size_used": X_train.shape[1],
        "accuracy": round(accuracy_score(y_test, preds), 4),
        "macro_f1": round(f1_score(y_test, preds, average="macro"), 4),
    })

probe_df = pd.DataFrame(probe_results)
probe_df.to_csv("../reports/tables/sentiment_probe.csv", index=False)
display(probe_df)

,tokenizer,vocab_size_used,accuracy,macro_f1
0,mBERT,3386,0.5911,0.4921
1,XLM-R,3907,0.5997,0.4889
2,AfroXLMR,3907,0.5997,0.4889
3,ABENA,3381,0.5940,0.4945
4,GhanaTok,1773,0.5968,0.5110
5,GhanaTok-Unigram,1893,0.5911,0.4866
